# Working with Object Detection Datasets

`datamaite` loads still-image object-detection datasets (COCO, YOLO,
VisDrone) into the neutral `ObjectDetectionDataset`, which satisfies the
object-detection protocol from
[MAITE](https://mit-ll-ai-technology.github.io/maite/) -- JATIC's common
interface for test-and-evaluation datasets and models -- and converts
between the formats with registered writers. This tutorial builds tiny synthetic datasets in temp
directories -- it runs anywhere, with no downloads.

```bash
pip install "datamaite[maite,od]"
```

The `od` extra provides OpenCV for image decode; `maite` provides the
protocol check (see [installation](../getting-started/installation.md)).

## Who this is for and where it fits

**Persona: T&E engineer or model developer working with object-detection
data.**

Evaluating a detector starts with getting the labeled test data -- often
delivered as COCO, YOLO, or VisDrone -- into a form your evaluation tools
understand. Because `datamaite` loads each format into a MAITE object-detection
dataset, any MAITE-aware T&E tool can consume the result without
format-specific glue. This tutorial covers that data-preparation step of the
evaluation workflow: load a detection dataset, inspect it, and convert between
formats when a downstream tool or training pipeline needs a different one.

In [ ]:
import datamaite

# major.minor.patch only, so the docs don't show a hatch-vcs dev suffix in CI
print(f"datamaite {'.'.join(datamaite.__version__.split('.')[:3])}")

## Build a tiny COCO dataset

A COCO root is `annotations/instances.json` plus the referenced images.
Sparse category ids (`1`/`3`) are preserved, and images decode only on
sample access.

In [ ]:
import atexit
import json
import shutil
import tempfile
from pathlib import Path

import cv2
import numpy as np

from datamaite import load_od

coco_root = Path(tempfile.mkdtemp(prefix="dm_coco_"))
atexit.register(shutil.rmtree, coco_root, ignore_errors=True)  # removed when the kernel exits
(coco_root / "annotations").mkdir(parents=True, exist_ok=True)
for name, shade in (("a.jpg", 60), ("b.jpg", 200)):
    cv2.imwrite(str(coco_root / name), np.full((48, 64, 3), shade, np.uint8))

instances = {
    "categories": [
        {"id": 1, "name": "person", "supercategory": "person"},
        {"id": 3, "name": "car", "supercategory": "vehicle"},
    ],
    "images": [
        {"id": 100, "file_name": "a.jpg", "width": 64, "height": 48},
        {"id": 101, "file_name": "b.jpg", "width": 64, "height": 48},
    ],
    "annotations": [
        {"id": 900, "image_id": 100, "category_id": 1, "bbox": [10, 20, 30, 20], "area": 600, "iscrowd": 0},
        {"id": 901, "image_id": 100, "category_id": 3, "bbox": [5, 5, 10, 10], "area": 100, "iscrowd": 0},
        {"id": 902, "image_id": 101, "category_id": 3, "bbox": [0, 0, 20, 15], "area": 300, "iscrowd": 0},
    ],
}
(coco_root / "annotations" / "instances.json").write_text(json.dumps(instances))

ds = load_od(coco_root, dataset_format="coco")
print("samples/detections :", ds.sample_count, "/", ds.num_detections)
print("index2label        :", ds.index2label())

## Inspect a sample

Indexing decodes the image lazily (CHW `uint8`) and builds an
`ObjectDetectionTarget`: boxes in xyxy, labels carrying the source category
ids. The dataset satisfies the MAITE object-detection protocol.

In [ ]:
from maite.protocols import object_detection as od_protocols

image, target, metadata = ds[0]
print("image shape (CHW):", image.shape, image.dtype)
print("target.boxes xyxy:", np.asarray(target.boxes).tolist())
print("target.labels    :", np.asarray(target.labels).tolist())
print("metadata         :", metadata)
print("MAITE OD Dataset :", isinstance(ds, od_protocols.Dataset))

## Convert between OD formats

`validate()` covers HMIE only today, so detection datasets go straight to
conversion: a COCO round-trip preserves the dataset, and COCO → YOLO is a
format change within the same task.

In [ ]:
from datamaite import convert

rt_root = coco_root.parent / (coco_root.name + "_rt")
atexit.register(shutil.rmtree, rt_root, ignore_errors=True)
convert(coco_root, rt_root, input_format="coco", output_format="coco")
reloaded = load_od(rt_root, dataset_format="coco")
print(
    "round-trip preserved:",
    reloaded.sample_count == ds.sample_count
    and reloaded.num_detections == ds.num_detections
    and reloaded.index2label() == ds.index2label(),
)

yolo_out = coco_root.parent / (coco_root.name + "_as_yolo")
atexit.register(shutil.rmtree, yolo_out, ignore_errors=True)
convert(coco_root, yolo_out, input_format="coco", output_format="yolo")
print("COCO -> YOLO detections:", load_od(yolo_out, dataset_format="yolo").num_detections)

## Load YOLO detection data

YOLO detection (`images/` + `labels/*.txt` + `data.yaml`) loads into the
same `ObjectDetectionDataset`. Because the YOLO family also serves
classification, generic `load`/`convert` calls need `task="od"`; `load_od`
sets it for you.

In [ ]:
yolo_root = Path(tempfile.mkdtemp(prefix="dm_yolo_od_"))
atexit.register(shutil.rmtree, yolo_root, ignore_errors=True)
(yolo_root / "images" / "train").mkdir(parents=True, exist_ok=True)
(yolo_root / "labels" / "train").mkdir(parents=True, exist_ok=True)
cv2.imwrite(str(yolo_root / "images" / "train" / "a.png"), np.full((48, 64, 3), 90, np.uint8))
# label line: class cx cy w h (normalized)
(yolo_root / "labels" / "train" / "a.txt").write_text("1 0.5 0.5 0.4 0.6\n")
(yolo_root / "data.yaml").write_text("path: .\ntrain: images/train\nnames: ['cat', 'dog']\n")

ds_yolo = load_od(yolo_root, dataset_format="yolo")
print("index2label :", ds_yolo.index2label())
print("detections  :", ds_yolo.num_detections)

## Load a VisDrone directory

VisDrone static images (`images/` + `annotations/*.txt`) load one sample per
image with the fixed 12-class VisDrone taxonomy, and `write(...,
output_format="visdrone")` emits the official VisDrone-DET split-root layout
the loader reads back -- category ids are preserved, so a VisDrone round-trip
is exact. The same directory also serves classification as per-box crops; see
[Working with IC Datasets](Working_with_IC_datasets.ipynb).

In [ ]:
vd_root = Path(tempfile.mkdtemp(prefix="dm_visdrone_"))
atexit.register(shutil.rmtree, vd_root, ignore_errors=True)
(vd_root / "images").mkdir(parents=True)
(vd_root / "annotations").mkdir()
cv2.imwrite(str(vd_root / "images" / "0000001.jpg"), np.full((60, 80, 3), 120, np.uint8))
# rows: left,top,width,height,score,category,truncation,occlusion (1=pedestrian, 4=car)
(vd_root / "annotations" / "0000001.txt").write_text("10,10,20,15,1,1,0,0\n40,20,25,30,1,4,0,0\n")

ds_vd = load_od(vd_root, dataset_format="visdrone")
img, tgt, _ = ds_vd[0]
print("samples:", ds_vd.sample_count, "| detections:", ds_vd.num_detections, "| image", img.shape)

In [ ]:
from datamaite import write

vd_out = vd_root.parent / (vd_root.name + "_out")
atexit.register(shutil.rmtree, vd_out, ignore_errors=True)
write(ds_vd, vd_out, output_format="visdrone")  # OD writer: dispatches on ds_vd.task

split_root = vd_out / "VisDrone2019-DET-train"  # the official DET split-root layout
print((split_root / "annotations" / "0000001.txt").read_text(), end="")
reloaded_vd = load_od(split_root, dataset_format="visdrone")
print(
    "round-trip preserved:",
    reloaded_vd.sample_count == ds_vd.sample_count and reloaded_vd.num_detections == ds_vd.num_detections,
)

## Where to next

[Working with IC Datasets](Working_with_IC_datasets.ipynb) covers the
classification surface (including crops from this same VisDrone layout);
[Working with FMV Datasets](Working_with_FMV_datasets.ipynb) covers video;
the [architecture reference](../reference/architecture.md) documents the
loader/writer registry.

The temp directories are removed when the kernel exits; the cell below
cleans up eagerly.

In [ ]:
for p in (coco_root, rt_root, yolo_out, yolo_root, vd_root):
    shutil.rmtree(p, ignore_errors=True)
print("removed temp datasets")

<hr>